# Import Modules

In [ ]:
from utils import *
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Tahoma'
plt.rcParams['axes.unicode_minus'] = False

# Validation

In [ ]:
issues_df = run_validation()

# Clean Data

In [ ]:
dfs_district = []
dfs_partylist = []
dfs_referendum = []

for file_config in OCR_FILES:
    total_pages = file_config["pages"]
    if "district" in file_config["path"]: 
        for page_num in range(1, total_pages, 2):
            df_dict = load_ocr_clean_data(file_config["path"], page_num)
            dfs_district.append(df_dict)
            
    elif "partylist" in file_config["path"]:
        for page_num in range(1, total_pages, 4):
            df_dict = load_ocr_clean_data(file_config["path"], page_num)
            dfs_partylist.append(df_dict)
            
    elif "referendum" in file_config["path"]:
        for page_num in range(1, total_pages, 2):
            df_dict = load_ocr_clean_data(file_config["path"], page_num)
            dfs_referendum.append(df_dict)

In [ ]:
district_summary, district_scores = flatten_district(
    dfs_district
)
partylist_summary, partylist_scores = flatten_partylist(
    dfs_partylist
)
referendum_summary, referendum_scores = flatten_referendum(
    dfs_referendum
)

district_scores["คะแนน"] = pd.to_numeric(district_scores["คะแนน"], errors="coerce").fillna(0).astype(int)
partylist_scores["คะแนน"] = pd.to_numeric(partylist_scores["คะแนน"], errors="coerce").fillna(0).astype(int)
referendum_scores["คะแนน"] = pd.to_numeric(referendum_scores["คะแนน"], errors="coerce").fillna(0).astype(int)

In [ ]:
district_summary.tail(5)

In [ ]:
district_scores.tail(10)

In [ ]:
partylist_summary.tail(5)

In [ ]:
partylist_scores.tail(10)

In [ ]:
referendum_summary.tail(5)

In [ ]:
referendum_scores.tail(5)

In [ ]:
district_summary.to_csv("flatten_result/district_summary.csv")
district_scores.to_csv("flatten_result/district_scores.csv")
partylist_summary.to_csv("flatten_result/partylist_summary.csv")
partylist_scores.to_csv("flatten_result/partylist_scores.csv")
referendum_summary.to_csv("flatten_result/referendum_summary.csv")
referendum_scores.to_csv("flatten_result/referendum_scores.csv")

# Insights

## District

### 1. คะแนนรวมแต่ละผู้สมัครจากทุกหน่วย

In [ ]:
d1 = (district_scores.groupby(["ชื่อผู้สมัคร", "พรรค"], as_index=False)["คะแนน"]
        .sum().sort_values("คะแนน", ascending=False).reset_index(drop=True))
d1.index += 1
d1["% คะแนน"] = (d1["คะแนน"] / d1["คะแนน"].sum() * 100).round(2)

display(d1)

fig, ax = plt.subplots(figsize=(10, max(4, len(d1) * 0.5)))
colors = ["red" if i == 0 else "blue" for i in range(len(d1))]
ax.barh(d1["ชื่อผู้สมัคร"] + " (" + d1["พรรค"] + ")", d1["คะแนน"], color=colors)
ax.set_xlabel("คะแนน")
ax.set_title("คะแนนรวมทุกหน่วย")
ax.invert_yaxis()
for i, v in enumerate(d1["คะแนน"]):
    ax.text(v + 10, i, f"{v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

### 2. จำนวนหน่วยที่ผู้สมัครแต่ละคนชนะ

In [ ]:
unit_winner = district_scores.loc[
    district_scores.groupby("unit_key")["คะแนน"].idxmax(), ["unit_key", "ชื่อผู้สมัคร", "พรรค"]
]
d2 = (unit_winner.groupby(["ชื่อผู้สมัคร", "พรรค"])
      .size().reset_index(name="หน่วยที่ชนะ")
      .sort_values("หน่วยที่ชนะ", ascending=False).reset_index(drop=True))
d2.index += 1

display(d2)

fig, ax = plt.subplots(figsize=(10, max(4, len(d2) * 0.5)))
colors = ["red" if i == 0 else "blue" for i in range(len(d2))]
ax.barh(d2["ชื่อผู้สมัคร"] + " (" + d2["พรรค"] + ")", d2["หน่วยที่ชนะ"], color=colors)
ax.set_xlabel("จำนวนหน่วย")
ax.set_title("จำนวนหน่วยที่ชนะ")
ax.invert_yaxis()
for i, v in enumerate(d2["หน่วยที่ชนะ"]):
    ax.text(v + 0.3, i, str(v), va="center", fontsize=8)
plt.tight_layout()
plt.show()

### 3. อัตราการมาใช้สิทธิเลือกตั้งสส.เขตรายตำบล

In [ ]:
district_summary["turnout"] = (
    (district_summary["บัตรดี"] + district_summary["บัตรเสีย"] + district_summary["ไม่เลือกผู้ใด"])
        / district_summary["จำนวนบัตรทั้งหมด"]
).replace([np.inf, -np.inf], np.nan)

turnout_by_tambon = (
    district_summary.groupby("ตำบล").apply(lambda x: (
        (x["บัตรดี"] + x["บัตรเสีย"] + x["ไม่เลือกผู้ใด"]).sum()
        / x["จำนวนบัตรทั้งหมด"].sum()
    ))
    .reset_index().rename(columns={0: "turnout"}).sort_values("turnout", ascending=False)
)

print(f"Turnout เฉลี่ยทั้งหมด: {district_summary['turnout'].mean():.1%}")
print(f"Turnout สูงสุด: {district_summary['turnout'].max():.1%}")
print(f"Turnout ต่ำสุด (หน่วย): {district_summary['turnout'].min():.1%}")

fig, ax = plt.subplots()
ax.barh(turnout_by_tambon["ตำบล"], turnout_by_tambon["turnout"] * 100, color="green")
ax.set_xlabel("Turnout (%)")
ax.set_title("Voter Turnout รายตำบล")
ax.axvline(district_summary["turnout"].mean() * 100, color="red", linestyle="--", label="ค่าเฉลี่ย")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 4. การกระจายคะแนนแต่ละพรรครายตำบล

In [ ]:
d4 = district_scores.groupby(["ตำบล", "พรรค"], as_index=False)["คะแนน"].sum()
d4_pivot = d4.pivot_table(index="ตำบล", columns="พรรค", values="คะแนน", fill_value=0)
d4_pivot.plot(kind="bar", figsize=(14, 6), title="การกระจายคะแนนสส.เขตรายตำบล (แยกพรรค)")

display(d4_pivot)

plt.ylabel("คะแนน")
plt.xlabel("ตำบล")
plt.xticks(rotation=45, ha="right")
plt.legend(title="พรรค", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 5. บัตรเสียรายหน่วย

In [ ]:
district_summary["bad_ballot_pct"] = (
    district_summary["บัตรเสีย"] / district_summary["จำนวนบัตรทั้งหมด"]
).replace([np.inf, -np.inf], np.nan)

threshold = district_summary["bad_ballot_pct"].mean() + 2 * district_summary["bad_ballot_pct"].std()
anomaly = district_summary[district_summary["bad_ballot_pct"] > threshold].sort_values("bad_ballot_pct", ascending=False)

print(f"เกณฑ์ผิดปกติ (mean + 2σ): {threshold:.2%}")
print(f"จำนวนหน่วยที่น่าสงสัย: {len(anomaly)}")
display(anomaly[["unit_key", "ตำบล", "อำเภอ", "บัตรเสีย", "จำนวนบัตรทั้งหมด", "bad_ballot_pct"]])

fig, ax = plt.subplots()
ax.hist(district_summary["bad_ballot_pct"].dropna() * 100, bins=30, color="orange", edgecolor="white")
ax.axvline(threshold * 100, color="red", linestyle="--", label=f"เกณฑ์ ({threshold:.1%})")
ax.set_xlabel("% บัตรเสีย")
ax.set_ylabel("จำนวนหน่วย")
ax.set_title("Distribution บัตรเสีย")
ax.legend()
plt.tight_layout()
plt.show()

### 6. พรรคที่ชนะในแต่ละตำบล

In [ ]:
total_votes = district_scores.groupby('ตำบล')['คะแนน'].sum().reset_index(name='total_votes')
party_votes = district_scores.groupby(['ตำบล', 'พรรค'])['คะแนน'].sum().reset_index()

merged = party_votes.merge(total_votes, on='ตำบล')
merged['สัดส่วนคะแนน'] = merged['คะแนน'] / merged['total_votes']

top_party = merged.sort_values('สัดส่วนคะแนน', ascending=False).drop_duplicates('ตำบล')

party_colors = {p: c for p, c in zip(district_scores['พรรค'].unique(),plt.cm.tab10.colors)}

colors = top_party['พรรค'].map(party_colors)

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(top_party)), top_party['สัดส่วนคะแนน'], color=colors)

for i, v in enumerate(top_party['สัดส่วนคะแนน']):
    plt.text(i, v + 0.005, f"{v*100:.1f}%", ha='center', va='bottom', fontsize=8)

plt.xticks(range(len(top_party)), top_party['ตำบล'], rotation=45)
plt.title('พรรคที่ได้คะแนนสูงสุดในแต่ละตำบล')
plt.xlabel('ตำบล')
plt.ylabel('สัดส่วนคะแนน')

handles = [plt.Rectangle((0,0),1,1, color=c) for c in party_colors.values()]
plt.legend(handles, party_colors.keys(), title='พรรค')

plt.tight_layout()
plt.show()

### 7. พื้นที่ฐานเสียงแข็งและพื้นที่ที่ต้องพัฒนา

In [ ]:
TOP = 5

margin_df = district_scores.copy()
margin_df = margin_df.sort_values(["unit_key", "คะแนน"], ascending=[True, False])

def calc_margin(group):
    group = group.sort_values("คะแนน", ascending=False).reset_index(drop=True)
    if len(group) < 2:
        return pd.Series({"winner": group.loc[0, "ชื่อผู้สมัคร"], "พรรค": group.loc[0, "พรรค"],
                          "winner_score": group.loc[0, "คะแนน"], "margin": group.loc[0, "คะแนน"], "margin_pct": 1.0})
    margin = group.loc[0, "คะแนน"] - group.loc[1, "คะแนน"]
    total = group["คะแนน"].sum()
    return pd.Series({"winner": group.loc[0, "ชื่อผู้สมัคร"], "พรรค": group.loc[0, "พรรค"],
                      "winner_score": group.loc[0, "คะแนน"], "margin": margin, "margin_pct": margin / total})

unit_margin = district_scores.groupby("unit_key").apply(calc_margin, include_groups=False).reset_index()
unit_margin = unit_margin.merge(district_scores[["unit_key", "หน่วย", "ตำบล", "อำเภอ"]].drop_duplicates(), on="unit_key")

parties = district_scores["พรรค"].unique()

for party in parties:
    own = unit_margin[unit_margin["พรรค"] == party].copy()
    lost = unit_margin[unit_margin["พรรค"] != party].copy()
    own_sorted = own.sort_values("margin_pct", ascending=False).head(TOP).reset_index(drop=True)
    lost_sorted = lost.sort_values("margin_pct", ascending=False).head(TOP).reset_index(drop=True)

    if own_sorted.empty and lost_sorted.empty:
        continue

    n_rows = max(len(own_sorted), len(lost_sorted), 1)
    fig_height = max(4, n_rows * 1.2)

    fig, axes = plt.subplots(1, 2, figsize=(12, fig_height))
    fig.suptitle(f"พรรค{party}", fontweight="bold")

    if not own_sorted.empty:
        own_sorted = own_sorted.dropna(subset=["ตำบล", "หน่วย"]).reset_index(drop=True)
        axes[0].barh(own_sorted["ตำบล"] + " หน่วย " + own_sorted["หน่วย"],
                     own_sorted["margin_pct"] * 100, color="green")
        axes[0].set_ylim(len(own_sorted) - 0.5, -0.5)
        for i, v in enumerate(own_sorted["margin_pct"]):
            axes[0].text(v * 100 + 0.3, i, f"{v:.1%}", va="center")
        axes[0].set_xlim(0, own_sorted["margin_pct"].max() * 100 * 1.15)

    axes[0].set_xlabel("Margin (%)")
    axes[0].set_title(f"Top {TOP} ฐานเสียงแข็ง")

    if not lost_sorted.empty:
        lost_sorted = lost_sorted.dropna(subset=["ตำบล", "หน่วย"]).reset_index(drop=True)
        axes[1].barh(lost_sorted["ตำบล"] + " หน่วย " + lost_sorted["หน่วย"],
                     lost_sorted["margin_pct"] * 100, color="red")
        axes[1].set_ylim(len(lost_sorted) - 0.5, -0.5)
        for i, v in enumerate(lost_sorted["margin_pct"]):
            axes[1].text(v * 100 + 0.3, i, f"{v:.1%}", va="center")
        axes[1].set_xlim(0, lost_sorted["margin_pct"].max() * 100 * 1.15)

    axes[1].set_xlabel("Margin (%)")
    axes[1].set_title(f"Top {TOP} พื้นที่ที่แพ้ห่างที่สุด")

    plt.tight_layout()
    plt.show()